# KTP database exploration

Testing access to Airtable dataset via the airtable HTTP API. Requires a valid api_token.

https://airtable.com/appQbIS18Qc8GskMC/api/docs#curl/table:quotes:list

Looks like we can only load 100 rows at a time. We need to use pagination and joins to extract all data from the REST API.

https://docs.google.com/presentation/d/1SyLoZid1-M3FXTjwZlmRxaPyBe9iZ1Q0m8T3qdjMKpw/edit

https://vizzuality.blogin.co/single-post.php?id=63837

https://towardsdatascience.com/basic-nlp-on-the-texts-of-harry-potter-sentiment-analysis-1b474b13651d

In [ ]:
import getpass
import requests
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

In [ ]:
api_token = getpass.getpass()

In [ ]:
url = "https://api.airtable.com/v0/appQbIS18Qc8GskMC/Quotes"

In [ ]:
params = {'api_key': api_token}
r = requests.get(url=url, params=params)

In [ ]:
tmp = []
for row in r.json().get('records'):
    tmp.append(row.get('fields'))
df = pd.DataFrame(tmp)

In [ ]:
len(df)

In [ ]:
df.head()

Interesting, looks like the tags are associated to a uniqe ID, it is that ID that is returned, not the string label of the tags. Will need to create a mapping to the strings in Python.

In [ ]:
df.tail()

## Loading data directly

Might be easier just to grab the data directly outside of the API

In [ ]:
csv = pd.read_csv("/Users/Ben/Downloads/Quotes-Grid.csv")

In [ ]:
csv.head()

In [ ]:
len(csv)

In [ ]:
bag_of_tags = []
for tag in csv['Sub-Tags']:
    #print(tag, tag == tag)
    if tag == tag:
        tmp = tag.split(',')
        for item in tmp:
            bag_of_tags.append(item)

In [ ]:
# Unique sub-tags
set(bag_of_tags)

In [ ]:
plt.figure(figsize=(10,10))
plt.hist(bag_of_tags)
plt.title("Number of occurences of each sub-tag in database")
_ = plt.xticks(rotation=90)

In [ ]:
#!pip install vaderSentiment

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [ ]:
def parse_platform_tag(platform):
    """Nan values are replaced with None"""
    if platform == platform:
        tmp = platform.split(',')
        if 'Exit Interviews' in tmp:
            if len(tmp) == 1:
                return tmp[0]
            #print(tmp)
            tmp = tmp[1]
        else:
            tmp = tmp[0]
        return tmp
    else:
        return None

In [ ]:
sentences = {}
analyzer = SentimentIntensityAnalyzer()
for row in range(len(csv)):
    try:
        sentence = csv.iloc[row]['Quotes']
        quote_id = csv.iloc[row]['quote_id']
        platform = parse_platform_tag(csv.iloc[row]['Platform'])
        vs = analyzer.polarity_scores(sentence)
        sentences[quote_id] = {'sentence': sentence, 'vs': vs, 'platform': platform}
    except:
        pass

In [ ]:
print(f"extracted senteiment for {len(sentences)} out of {len(csv)} quotes")

In [ ]:
# We can now access each sentence using its quote_id as an index

sentences[1]

In [ ]:
# The unique platforms

all_platforms = []
for row in range(len(csv)):
    all_platforms.append(parse_platform_tag(csv.iloc[row]['Platform']))
    
    
unique_platforms = set(all_platforms)
unique_platforms

In [ ]:
# Calcuate sentiment score across platforms

for platform in unique_platforms:
    if platform and platform is not 'Exit Interviews':
        print(platform)

In [ ]:
def wc(sentence):
    word_count = len(sentence.split(' '))
    return word_count

In [ ]:
#platform

platform_scores = {}
quote_compound_scores = {}

for platform in unique_platforms:
    if platform and platform != 'Exit Interviews':
        print(f"working on {platform}")
        words_per_quote = []
        positive = []
        negative = []
        neutral = []
        compound = []
        for quote_id in sentences:
            if sentences[quote_id].get('platform') == platform:
                #print(sentences[sentence])
                words_per_quote.append(wc(sentences[quote_id].get('sentence')))
                positive.append(sentences[quote_id].get('vs').get('pos'))
                negative.append(sentences[quote_id].get('vs').get('neg'))
                neutral.append(sentences[quote_id].get('vs').get('neu'))
                compound.append(sentences[quote_id].get('vs').get('compound'))
        quote_compound_scores[platform] = compound
        platform_scores[platform] = {   'avg_compound': np.average(compound),
                                        'std_compound': np.std(compound),
                                        'avg_word_count': np.average(words_per_quote),
                                        'std_word_count': np.std(words_per_quote),
                                        'total_words': np.sum(words_per_quote),
                                        'avg_positivity': np.average(positive),
                                        'std_positivity': np.std(positive),
                                        'avg_neutrality': np.average(neutral),
                                        'std_neutrality': np.std(neutral),
                                        'avg_negativity': np.average(negative),
                                        'std_negativity': np.std(neutral)
                                    }

In [ ]:
platform_scores

In [ ]:
platforms = []
word_counts = []
word_counts_sigma = []
total_words = []
for p in platform_scores:
    platforms.append(p)
    word_counts.append(platform_scores[p].get('avg_word_count'))
    word_counts_sigma.append(platform_scores[p].get('std_word_count'))
    total_words.append(platform_scores[p].get('total_words'))

In [ ]:
# First a simple analysis, how many words do we have per quote?

plt.figure(figsize=(10,5))
plt.bar(platforms, word_counts,yerr=word_counts_sigma,
           align='center',
           alpha=0.5,
           ecolor='black',
           capsize=10)
plt.title("Average number of words per quote per platform")
plt.ylabel("Number of words per quote")
_ =plt.xticks(rotation=90)

In [ ]:
# Next, how many total words per platform?

plt.figure(figsize=(10,5))
plt.bar(platforms, total_words, alpha=0.5)
plt.title("Total number of words across all interviews per platform")
plt.ylabel("Number of words")
_ =plt.xticks(rotation=90)

In [ ]:
platform_scores

In [ ]:
platforms = []
positvity = []
positivity_sigma = []
word_counts_sigma = []
neutrality = []
neutrality_sigma = []
negativity = []
negativity_sigma = []
compound = []
compound_sigma = []
for p in platform_scores:
    platforms.append(p)
    compound.append(platform_scores[p].get('avg_compound'))
    compound_sigma.append(platform_scores[p].get('std_compound'))
    positvity.append(platform_scores[p].get('avg_positivity'))
    positivity_sigma.append(platform_scores[p].get('std_positivity'))
    neutrality.append(platform_scores[p].get('avg_neutrality'))
    neutrality_sigma.append(platform_scores[p].get('std_neutrality'))
    negativity.append(platform_scores[p].get('avg_negativity'))
    negativity_sigma.append(platform_scores[p].get('std_negativity'))

In [ ]:
platforms

In [ ]:
data = [quote_compound_scores[plat] for plat in platforms]

In [ ]:
plt.figure(figsize=(10,10))
_= plt.boxplot(data, notch=True, labels= platforms, bootstrap=10000)
_ =plt.xticks(rotation=90)
_ =plt.title('Distribution of sentiment by platform')
_=plt.ylabel("sentiment score")

Ok this is starting to get interesting. Seems that TRASE is least popular by sentiment, followed by climate watch. GFW and Fishing watch seem to have about the same performance, but Half Earth seems to be the platform with the widest spread of sentiment and the highest median sentiment.

### Sentiment broken by positivity 

In [ ]:

width = 0.35

plt.figure(figsize=(20,10))
plt.title('Sentiment by platform')
plt.subplot(1, 3, 1)
plt.bar(platforms, positvity, width, yerr=positivity_sigma, color='green', alpha=0.5)
plt.ylabel('Positive Sentiment score')
plt.xticks(platforms)
_ =plt.xticks(rotation=90)


plt.subplot(1, 3, 2)
plt.bar(platforms, neutrality, width, yerr=neutrality_sigma, color='blue', alpha=0.5)
plt.ylabel('Neutral Sentiment score')
plt.xticks(platforms)
_ =plt.xticks(rotation=90)


plt.subplot(1, 3, 3)
plt.bar(platforms, negativity, width, yerr=negativity_sigma, color='red', alpha=0.5)
plt.ylabel('Negative Sentiment score')
plt.xticks(platforms)
_ =plt.xticks(rotation=90)


plt.show()

In [ ]:
# Viewing the integrated sentiment scores per platform

width = 0.35

p1 = plt.bar(platforms, negativity, width, yerr=negativity_sigma, color='red', alpha=0.5)
p2 = plt.bar(platforms, neutrality, width, yerr=neutrality_sigma, color='blue', alpha=0.5, bottom=negativity)
p3 = plt.bar(platforms, positvity, width, yerr=neutrality_sigma, color='green', alpha=0.5, bottom=np.array(negativity)+np.array(neutrality))

plt.ylabel('Sentiment score')
plt.title('sentiment by platform')
plt.xticks(platforms)
_ =plt.xticks(rotation=90)

plt.show()

Difficult to see any appreciable diffrence. Certainly none at the aggregated level beyond the ?1 standard deviation level. (Although I am not sure if that is meaninfgul or not in this case.)

### Let's start looking at ranking from sentiment and see if it lines up with the self-reported prefrences across the platform

In [ ]:
positive_rank = {}
for plat, pos in zip(platforms, positvity):
    positive_rank[plat] = pos

In [ ]:
positive_rank

In [ ]:
import operator

In [ ]:
sorted_platforms = sorted(positive_rank.items(), key=operator.itemgetter(1))

In [ ]:
# sorted from least positive to most positive
sorted_platforms

## Break the quotes into primary Emotions

We will be using the [NRC Word-Emotion Association Lexicon aka NRC Emotion Lexicon aka EmoLex](http://sentiment.nrc.ca/lexicons-for-research/), obtained under the [research liscence](http://saifmohammad.com/WebPages/AccessResource.htm).

**Crowdsourcing a Word-Emotion Association Lexicon, Saif Mohammad and Peter Turney, Computational Intelligence, 29 (3), 436-465, 2013**


We will need to:
1. Remove puncutation and tokenize sentences
2. Lemmatize the individual words to return them to the dictionary root ( https://stackabuse.com/python-for-nlp-tokenization-stemming-and-lemmatization-with-spacy-library/ )
3. Lookup the words in the emotion Lexicon and grab the emotion class where given

*nb. need to be on python 3.6 for this library*
```
$ conda install -c conda-forge spacy
$ python -m spacy download en
$ python
import spacy
sp = spacy.load('en_core_web_sm')
```

### todo

We need to create a small dictionary of problematic domain words, for example, when people talk about goods in TRASE that should not be parsed into the Lexical root (good) as it is not talking about an emotion.
Removing the false positives that come from domain specific word roots will increase the signal from people talking about the site itself. Something particularly important when we are dealing with small sample sizes, but that may simply contribute to noise if we were dealing with automatic surveys with thosands of respondants.

In [ ]:
from pprint import pprint

In [ ]:
# Load the libs required and config 
import spacy
sp = spacy.load('en_core_web_sm')

In [ ]:
# Load the emotion Lexicon
emotions = pd.read_csv("/Users/Ben/Downloads/NRC-Sentiment-Emotion-Lexicons/NRC-Sentiment-Emotion-Lexicons/NRC-Emotion-Lexicon-v0.92/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt",
                      delimiter="	", header=None,names=['word','emotion','e_flag'])
print(f"Loaded {len(emotions):,g} rows of data from the emotion Lexicon")

In [ ]:
# example of how words have been tagged in the Lexicon
emotions[emotions['word'] == 'excited']

In [ ]:
def emotion_analysis(text, verbose=False):
    if type(text) != str:
        raise ValueError("emotion_analysis function must be given a string as an input")
    # Lemmatization using spaCy
    emotion_dic = {}
    for word in sp(text):
        #print(word.text, word.lemma_)
        lemma = word.lemma_
        tmp_df = emotions[emotions['word'] == lemma]
        if len(tmp_df) == 10:
            elist = list(tmp_df[tmp_df['e_flag'] == 1]['emotion'].values)
            if len(elist) > 0:
                emotion_dic[lemma] = elist
                if verbose: print(f'?: {word.text} | {lemma}  |  {elist}')  
            
        else:
            if verbose: print(f"?: {word.text} | {lemma}")
            tmp_df = None
    if verbose: print(f'Lemmatized: {text}')
    if verbose: pprint(emotion_dic)
    return emotion_dic

In [ ]:
# example sentence
text = sentences[1].get('sentence')
d = emotion_analysis(text, verbose=True)

In [ ]:
# # Lemmatization using spaCy
# emotion_dic = {}
# for word in sp(sentences[2].get('sentence')):
#     #print(word.text, word.lemma_)
#     lemma = word.lemma_
#     tmp_df = emotions[emotions['word'] == lemma]
#     if len(tmp_df) == 10:
#         elist = list(tmp_df[tmp_df['e_flag'] == 1]['emotion'].values)
#         if len(elist) > 0:
#             emotion_dic[lemma] = elist
#         print(f'?: {word.text} | {lemma}  |  {elist}')
#     else:
#         print(f"?: {word.text} | {lemma}")
#         tmp_df = None

In [ ]:
# list(tmp_df[tmp_df['e_flag'] == 1]['emotion'].values)

In [ ]:
# lemma = 'learn'
# tmp_df = emotions[emotions['word'] == lemma]
# if len(tmp_df) == 10:
#     print(f'?: {lemma}')
# else
#     print(f"? {lemma}")